# 09 - Pass 4 analysis (CPU only, no GPU)

Turns the Pass 4 runs into the tables for the paper. It reads result folders only; it trains nothing. Runs locally in a few minutes (or on Kaggle with accelerator **None**). Code: `src/gainsplit/analysis_pass4.py`. Every test and contrast was fixed before the runs (vault: Experiment Plan, "Pass 4 plan"); scores are in points (macro-F1 0.882 = 88.2).

**Inputs:** `results/pass4/` (unzip `results_pass4.zip` there) plus the earlier passes, whose A1, A6 and A7 runs are the baselines.

## 1. Get the code and choose the folders

In [ ]:
import os, subprocess, sys
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    REPO_DIR = "/tmp/GainSplit"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Ariffurrhmn/GainSplit.git", REPO_DIR], check=True)
else:
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "experiments" else str(Path.cwd())
sys.path.insert(0, f"{REPO_DIR}/src")
import pandas as pd
pd.set_option("display.width", 200)

from gainsplit import analysis as an, analysis_pass4 as a4
R = Path(REPO_DIR) / "results"
RESULTS_DIRS = [R / p for p in ("pass1", "pass2a", "pass2b", "pass3", "pass4", "pass4_full100") if (R / p / "runs").exists()]
OUT_DIR = Path("/kaggle/working/analysis_pass4") if ON_KAGGLE else R / "analysis_pass4"
N_BOOT, RNG_SEED = 10000, 0
print([p.name for p in RESULTS_DIRS])
runs = an.load_runs(*RESULTS_DIRS)
print(len(runs), "runs")

## 2. Re-run check (read this first)
The A1 re-runs at 10% (tag `explain`) and the A1 / A7 re-runs at 1% (tag `mask`) repeat runs from Passes 1 and 2b with the new code. Their test scores and probabilities must be identical (`ok` = True). If not, the new code changed the old training path, and the Pass 4 contrasts against old baselines are not clean: stop and report.

In [ ]:
rr = a4.check_reruns(runs[runs["budget"].isin([1, 10])])
display(rr)
print("all identical:", bool(rr["ok"].all()))

## 3. Summary
Mean test macro-F1, cue-masked test macro-F1 and iSarcasmEval ROC-AUC (all points) per arm, budget and tag, with the sd over seeds.

In [ ]:
summ = a4.summary(runs)
summ

## 4. Main contrasts (paired bootstrap, 10,000 resamples)
Difference in test macro-F1 (points), 95% interval, two-sided p, and the Holm-adjusted p over all Pass 4 contrasts (robustness check, as in Passes 1-3). How to read them:
- **A10-A1 / A11-A7 / A11-A1:** do the pairs help, alone and on top of FGM?
- **A10-A12 pairing:** does the topic-matched pairing matter? If this is near 0, the claim is "generated text helps", not "counterfactual pairs help".
- **A13-A14 judge filter:** does the judge's choice help, at equal size and model mix?
- **A15-A1:** pairs only (no human labels) vs 10% human labels.

In [ ]:
con = a4.contrast_table(runs, N_BOOT, RNG_SEED)
con.round(4)

## 5. Outlet shortcut, transfer and explanations
- **Gap contrasts:** shortcut gap (random-mask F1 minus cue-mask F1) of a pair arm minus that of its baseline. Negative = the pair arm loses less when outlet words are hidden.
- **AUC contrasts:** iSarcasmEval ROC-AUC of a pair arm minus its baseline (points). In-domain reference from Pass 3: 65.8.
- **Explanations:** share of Integrated Gradients attribution on outlet cue words, A10 vs A1 at 10% (same 500 test headlines for every model and seed). `median_rel_completeness_gap` should be small (a few %): it checks that the attributions add up to the model's score difference.

In [ ]:
gaps = a4.gap_contrasts(runs, N_BOOT, RNG_SEED)
aucs = a4.auc_table(runs, N_BOOT, RNG_SEED)
expl = a4.load_explain(runs)
display(gaps.round(4)); display(aucs.round(4))
display(a4.explain_table(expl).round(4)); display(a4.explain_contrast_table(expl, N_BOOT, RNG_SEED).round(4))

## 6. Save every table (CSV + Markdown) to the output folder

In [ ]:
tables = a4.run_all(RESULTS_DIRS, OUT_DIR, N_BOOT, RNG_SEED)
print("written to", OUT_DIR, ":", [k for k, v in tables.items() if len(v)])